# SKIPP'D: 15-minute-ahead PV forecasting

Run each cell in Google Colab. Downloads the first 600 dataset records and sky images. Reconstructs the documented methodology; confirm metrics against the original research notebook.

In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2.2,<3" "scikit-learn>=1.4,<2" "requests>=2.31,<3" "Pillow>=10,<13" "matplotlib>=3.8,<4"

## Implementation
The following functions download SKIPP'D, derive mean central sky-image brightness, match exact 15-minute targets, fit two Random Forest regressors, and save results/figures.

In [ ]:
"""Reproducible 15-minute-ahead PV forecasting study using SKIPP'D.

This is a reconstructed, independently runnable implementation of the documented
experimental protocol. The original interactive Colab notebook was not supplied.
Run `python src/pv_forecast.py --out outputs` from the repository root.
"""
from __future__ import annotations

import argparse
import json
import time
from io import BytesIO
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from PIL import Image
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

API = "https://datasets-server.huggingface.co/rows"
DATASET = "solarbench/SKIPPD"
CONFIG = "default"
SPLIT = "train"
HORIZON = pd.Timedelta(minutes=15)


def request_json(session: requests.Session, offset: int, length: int) -> dict:
    """Fetch at most 100 consecutive viewer rows (HF API constraint)."""
    params = {"dataset": DATASET, "config": CONFIG, "split": SPLIT,
              "offset": offset, "length": length}
    for attempt in range(4):
        try:
            response = session.get(API, params=params, timeout=90)
            response.raise_for_status()
            return response.json()
        except requests.RequestException:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("Unreachable")


def request_image(session: requests.Session, url: str) -> Image.Image:
    for attempt in range(4):
        try:
            response = session.get(url, timeout=90)
            response.raise_for_status()
            # Hold a decoded copy so the source BytesIO may safely close.
            with Image.open(BytesIO(response.content)) as img:
                return img.convert("L").copy()
        except (requests.RequestException, OSError):
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)
    raise RuntimeError("Unreachable")


def center_brightness(image: Image.Image) -> float:
    """Mean grayscale brightness (0..255) in central 50% width and height."""
    gray = image.convert("L")
    width, height = gray.size
    center = gray.crop((int(width * 0.25), int(height * 0.25),
                        int(width * 0.75), int(height * 0.75)))
    return float(np.asarray(center, dtype=np.float64).mean())


def fetch_first_records(count: int = 600, *, session=None) -> pd.DataFrame:
    """Fetch first `count` SKIPP'D rows and compute brightness at each timestamp."""
    if count < 1:
        raise ValueError("count must be positive")
    session = session or requests.Session()
    rows = []
    for offset in range(0, count, 100):
        length = min(100, count - offset)
        response = request_json(session, offset, length)
        records = response["rows"]
        if len(records) != length:
            raise ValueError(f"Expected {length} rows at offset {offset}; got {len(records)}")
        for record in records:
            item = record["row"]
            img = item.get("image")
            src = img.get("src") if isinstance(img, dict) else None
            if not src:
                raise ValueError(f"Missing image URL at dataset row {offset + len(rows) % 100}")
            gray = request_image(session, src)
            rows.append({"time": item["time"], "pv": float(item["pv"]),
                         "center_brightness": center_brightness(gray)})
        print(f"Downloaded and processed {len(rows)} / {count} images", flush=True)
    df = pd.DataFrame(rows)
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").reset_index(drop=True)
    if df["time"].duplicated().any() or df.isna().any().any():
        raise ValueError("Input contains duplicate timestamps or missing values")
    return df


def make_forecasting_pairs(features: pd.DataFrame) -> pd.DataFrame:
    """Keep only timestamps with an EXACT +15-minute PV measurement."""
    power = features[["time", "pv"]].copy()
    power["time"] = pd.to_datetime(power["time"], utc=True)
    power = power.sort_values("time").reset_index(drop=True)
    if power["time"].duplicated().any():
        raise ValueError("Duplicate source timestamps")
    current = power.copy()
    current["target_time"] = current["time"] + HORIZON
    future = power.rename(columns={"time": "target_time", "pv": "actual_power"})
    pairs = current.merge(future, on="target_time", how="inner", validate="one_to_one")
    sky = features[["time", "center_brightness"]].copy()
    sky["time"] = pd.to_datetime(sky["time"], utc=True)
    pairs = pairs.merge(sky, on="time", how="left", validate="one_to_one")
    if pairs["center_brightness"].isna().any():
        raise ValueError("Missing brightness value at forecast issue time")
    pairs["hour"] = pairs["time"].dt.hour  # UTC, same as original notebook
    pairs["minute"] = pairs["time"].dt.minute
    return pairs.sort_values("time").reset_index(drop=True)


def chronological_split(pairs: pd.DataFrame, fraction: float = 0.8):
    """80/20 by issue time; drop train pairs with targets reaching test period."""
    n = int(len(pairs) * fraction)
    if not (0 < n < len(pairs)):
        raise ValueError("Not enough rows to create a train/test split")
    test = pairs.iloc[n:].copy().reset_index(drop=True)
    test_start = test["time"].iloc[0]
    train = pairs.iloc[:n].copy()
    train = train.loc[train["target_time"] < test_start].reset_index(drop=True)
    if train.empty or train["target_time"].max() >= test_start:
        raise ValueError("Invalid chronological split")
    return train, test, test_start


def errors(actual, predicted) -> dict:
    return {"mae_kw": float(mean_absolute_error(actual, predicted)),
            "rmse_kw": float(np.sqrt(mean_squared_error(actual, predicted)))}


def evaluate_models(pairs: pd.DataFrame) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train, test, test_start = chronological_split(pairs)
    a_features = ["pv", "hour", "minute"]
    b_features = a_features + ["center_brightness"]
    model_kwargs = dict(n_estimators=100, min_samples_leaf=3,
                        random_state=42, n_jobs=-1)
    model_a = RandomForestRegressor(**model_kwargs)
    model_b = RandomForestRegressor(**model_kwargs)
    model_a.fit(train[a_features], train["actual_power"])
    model_b.fit(train[b_features], train["actual_power"])
    results = test[["time", "target_time", "pv", "actual_power"]].copy()
    results["persistence_prediction"] = results["pv"]
    results["model_a_prediction"] = model_a.predict(test[a_features])
    results["model_b_prediction"] = model_b.predict(test[b_features])
    y = results["actual_power"]
    output = {
        "dataset": DATASET,
        "requested_source_rows": int(len(pairs) + 0),  # replaced by main() with 600
        "forecasting_pairs": int(len(pairs)),
        "training_rows": int(len(train)),
        "test_rows": int(len(test)),
        "boundary_crossing_pairs_dropped": int(int(len(pairs) * 0.8) - len(train)),
        "test_start_utc": str(test_start),
        "time_range_start_utc": str(pairs["time"].min()),
        "time_range_last_issue_utc": str(pairs["time"].max()),
        "experimental_settings": model_kwargs,
        "full_pairs_persistence": errors(pairs["actual_power"], pairs["pv"]),
        "common_test": {
            "persistence": errors(y, results["persistence_prediction"]),
            "random_forest_a": errors(y, results["model_a_prediction"]),
            "random_forest_b": errors(y, results["model_b_prediction"]),
        },
    }
    return output, train, test, results


def prepare_plot():
    plt.rcParams.update({"font.size": 9, "axes.titlesize": 10,
                         "savefig.dpi": 300})


def save_plot(path: Path):
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight", dpi=300)
    plt.close()


def plot_workflow(out: Path):
    fig, ax = plt.subplots(figsize=(8.5, 2.7))
    ax.axis("off")
    steps = ["SKIPP'D data\nPV + sky images", "15-min pairs\nUTC alignment",
             "Chronological\ntrain/test split", "Persistence\nRF-A / RF-B", "Common test\nMAE / RMSE"]
    for i, item in enumerate(steps):
        x = .10 + i * .20
        ax.text(x, .52, item, ha="center", va="center", fontsize=9,
                bbox=dict(boxstyle="round,pad=.65", facecolor="#EDF3F8", edgecolor="#576B7F"),
                transform=ax.transAxes)
        if i < len(steps) - 1:
            ax.annotate("", xy=(x + .14, .52), xytext=(x + .07, .52),
                        arrowprops=dict(arrowstyle="->", lw=1), xycoords="axes fraction")
    save_plot(out / "fig1_workflow.png")


def plot_outputs(features: pd.DataFrame, pairs: pd.DataFrame,
                 predictions: pd.DataFrame, metrics: dict, out: Path):
    out.mkdir(parents=True, exist_ok=True)
    prepare_plot()
    plot_workflow(out)

    plt.figure(figsize=(8, 3.1))
    plt.plot(features["time"], features["pv"], lw=1.1)
    plt.xlabel("Time (UTC)")
    plt.ylabel("PV power (kW)")
    plt.title("PV power profile: first 600 SKIPP'D records")
    plt.grid(alpha=.2)
    save_plot(out / "fig2_pv_profile.png")

    diff = pairs["actual_power"] - pairs["pv"]
    counts = [int((diff > 0).sum()), int((diff < 0).sum()), int((diff == 0).sum())]
    plt.figure(figsize=(5, 2.6))
    bars = plt.barh(["Increase", "Decrease", "Unchanged"], counts)
    plt.gca().invert_yaxis()
    plt.bar_label(bars, padding=3)
    plt.xlabel("Number of 15-minute pairs")
    save_plot(out / "fig3_power_change_directions.png")

    # Large ramp events: show local measured PV within a +/- 10 minute window
    fig, axes = plt.subplots(2, 1, figsize=(8, 5.4))
    for ax, idx, label in [
        (axes[0], diff.idxmax(), "Largest 15-minute increase"),
        (axes[1], diff.idxmin(), "Largest 15-minute decrease")
    ]:
        issue = pairs.loc[idx, "time"]
        target = pairs.loc[idx, "target_time"]
        subset = features.loc[features["time"].between(issue - pd.Timedelta(minutes=4),
                                                        target + pd.Timedelta(minutes=4))]
        ax.plot(subset["time"], subset["pv"], marker=".", ms=3, lw=1)
        ax.axvline(issue, ls="--", alpha=.45)
        ax.axvline(target, ls="--", alpha=.45)
        ax.set_ylabel("PV power (kW)")
        ax.set_title(f"{label}: {diff.loc[idx]:+.3f} kW")
        ax.grid(alpha=.2)
    axes[1].set_xlabel("UTC time")
    save_plot(out / "fig4_extreme_power_changes.png")

    def segmented_plot(frame, name, col):
        # Stop lines joining observations separated by long gaps.
        frame = frame.sort_values("target_time").reset_index(drop=True).copy()
        groups = (frame["target_time"].diff() > pd.Timedelta(minutes=3)).cumsum()
        for _, g in frame.groupby(groups):
            yield g

    plt.figure(figsize=(8, 3.2))
    first = True
    all_pairs = pairs.copy()
    all_pairs["persistence_prediction"] = all_pairs["pv"]
    for group in segmented_plot(all_pairs, "Persistence", "persistence_prediction"):
        plt.plot(group["target_time"], group["actual_power"], lw=1.2,
                 label="Actual PV power" if first else None)
        plt.plot(group["target_time"], group["persistence_prediction"], lw=1,
                 ls="--", label="Persistence" if first else None)
        first = False
    plt.ylabel("PV power (kW)")
    plt.xlabel("Target time (UTC)")
    plt.legend(fontsize=8)
    save_plot(out / "fig5_persistence_expanded.png")

    plt.figure(figsize=(8, 3.2))
    first = True
    for group in segmented_plot(predictions, "RF-A", "model_a_prediction"):
        plt.plot(group["target_time"], group["actual_power"], lw=1.2,
                 label="Actual PV power" if first else None)
        plt.plot(group["target_time"], group["model_a_prediction"], ls="--",
                 lw=1.1, label="Random Forest A" if first else None)
        first = False
    plt.xlabel("Target time (UTC)")
    plt.ylabel("PV power (kW)")
    plt.legend(fontsize=8)
    save_plot(out / "fig6_model_a_vs_actual.png")

    names = ["Persistence", "RF-A", "RF-B"]
    items = [metrics["common_test"][key] for key in
             ("persistence", "random_forest_a", "random_forest_b")]
    x = np.arange(3)
    fig, ax = plt.subplots(figsize=(6.0, 3.4))
    b1 = ax.bar(x - .18, [item["mae_kw"] for item in items], .35, label="MAE")
    b2 = ax.bar(x + .18, [item["rmse_kw"] for item in items], .35, label="RMSE")
    ax.set_xticks(x, names)
    ax.set_ylabel("Error (kW)")
    ax.legend(fontsize=8)
    ax.bar_label(b1, fmt="%.3f", fontsize=8, padding=2)
    ax.bar_label(b2, fmt="%.3f", fontsize=8, padding=2)
    ax.set_ylim(0, max(item["rmse_kw"] for item in items) * 1.19)
    save_plot(out / "fig7_three_model_metrics.png")

    plt.figure(figsize=(8, 3.4))
    first = True
    for group in segmented_plot(predictions, "comparison", "model_b_prediction"):
        plt.plot(group["target_time"], group["actual_power"], lw=1.3,
                 label="Actual PV power" if first else None)
        plt.plot(group["target_time"], group["model_a_prediction"], ls="--", lw=1.1,
                 label="Random Forest A" if first else None)
        plt.plot(group["target_time"], group["model_b_prediction"], ls=":", lw=1.2,
                 label="Random Forest B" if first else None)
        first = False
    plt.xlabel("Target time (UTC)")
    plt.ylabel("PV power (kW)")
    plt.legend(fontsize=8)
    save_plot(out / "fig8_model_a_model_b_vs_actual.png")


def run(*, records: int = 600, out: str | Path = "outputs", cached: str | None = None):
    destination = Path(out)
    destination.mkdir(parents=True, exist_ok=True)
    if cached:
        print(f"Loading cached image features: {cached}")
        features = pd.read_csv(cached)
        required = {"time", "pv", "center_brightness"}
        if not required.issubset(features.columns):
            raise ValueError(f"Cache must contain columns: {sorted(required)}")
        features["time"] = pd.to_datetime(features["time"], utc=True)
        features = features.sort_values("time").reset_index(drop=True)
    else:
        features = fetch_first_records(records)
    if features.isna().any().any() or features["time"].duplicated().any():
        raise ValueError("Source has nulls or duplicate timestamps")
    features.to_csv(destination / "image_features.csv", index=False)
    pairs = make_forecasting_pairs(features)
    pairs.to_csv(destination / "forecasting_pairs.csv", index=False)
    metrics, train, test, predictions = evaluate_models(pairs)
    metrics["requested_source_rows"] = int(len(features))
    metrics["source_start_utc"] = str(features["time"].min())
    metrics["source_end_utc"] = str(features["time"].max())
    metrics["unique_pacific_dates"] = int(features["time"].dt.tz_convert(
        "America/Los_Angeles").dt.date.nunique())
    predictions.to_csv(destination / "test_predictions.csv", index=False)
    with (destination / "metrics.json").open("w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=2, default=str)
    plot_outputs(features, pairs, predictions, metrics, destination / "figures")
    print("\nREPRODUCED METRICS")
    print(json.dumps(metrics, indent=2, default=str))
    if len(features) == 600:
        expected_counts = (526, 405, 106)
        observed_counts = (len(pairs), len(train), len(test))
        if observed_counts != expected_counts:
            print("\nWARNING: sample counts differ from the reported study:",
                  observed_counts, "vs", expected_counts)
        expected = {
            "persistence": (2.093, 3.124),
            "random_forest_a": (3.152, 4.031),
            "random_forest_b": (5.043, 6.057),
        }
        for name, (mae, rmse) in expected.items():
            observed = metrics["common_test"][name]
            if abs(observed["mae_kw"] - mae) > .015 or abs(observed["rmse_kw"] - rmse) > .015:
                print(f"WARNING: {name} differs from published-rounding targets. "
                      "Compare the original notebook and split settings.")
    return metrics


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--records", type=int, default=600)
    parser.add_argument("--out", type=str, default="outputs")
    parser.add_argument("--cached", type=str,
                        help="Existing CSV with time,pv,center_brightness; avoids redownloading images")
    args = parser.parse_args(argv)
    run(records=args.records, out=args.out, cached=args.cached)




## Execute experiment
Run once. To reuse your original CSV, uncomment the cached version instead (upload your file into Colab first).

In [ ]:
report = run(records=600, out="/content/solar_pv_outputs")
# Alternative (if already uploaded):
# report = run(out="/content/solar_pv_outputs", cached="/content/skippd_600_image_features.csv")

In [ ]:
from IPython.display import display, Image as IPImage
from pathlib import Path
for p in sorted(Path("/content/solar_pv_outputs/figures").glob("*.png")):
    print(p.name)
    display(IPImage(filename=str(p), width=640))

In [ ]:
import shutil
shutil.make_archive("/content/solar_pv_outputs", "zip", "/content/solar_pv_outputs")
print("Saved: /content/solar_pv_outputs.zip (download from Colab Files panel)")